In [1]:
import pandas as pd
import re
from ipynb.fs.defs.functions import clean_and_sort, validate_df

In [2]:
# CSV-Datei einlesen
df = pd.read_csv("../data/raw/2024/22517-02i.csv", sep=";", encoding="latin1", skiprows=6)
# Quelle: https://www.landesdatenbank.nrw.de/



In [3]:
# Überblick:
print(df.head(10))
print(df.info())

  Unnamed: 0 Unnamed: 1                            Unnamed: 2  \
0        NaN        NaN                                   NaN   
1        NaN        NaN                                   NaN   
2       2024         05                   Nordrhein-Westfalen   
3       2024        051          Düsseldorf, Regierungsbezirk   
4       2024      05111               Düsseldorf, krfr. Stadt   
5       2024      05112                 Duisburg, krfr. Stadt   
6       2024      05113                    Essen, krfr. Stadt   
7       2024      05114                  Krefeld, krfr. Stadt   
8       2024      05116          Mönchengladbach, krfr. Stadt   
9       2024      05117      Mülheim an der Ruhr, krfr. Stadt   

                              Insgesamt  \
0  In Anspruch genommene erzieh. Hilfen   
1                                Anzahl   
2                                311197   
3                                 89998   
4                                 11264   
5                         

In [4]:
# nach Kreisen und kreisfreien Städten und in Anspruch genommenen (35a) Hilfen filtern
df = df.rename(columns={"Unnamed: 2": "Name", "Eingliederungshilfe für seelisch behinderte junge Menschen § 35a SGB VIII": "Anzahl 35a Hilfen"})


# große kreisangehörige Städte filtern, in dem bei den Kreisen, in denen sich eine große Kommune befindet, ein entsprechendes Flag gesetzt wird
grosse_kommunen = "Arnsberg, Bergheim, Bergisch Gladbach, Bocholt, Castrop-Rauxel, Detmold, Dinslaken, Dormagen, Dorsten, Düren, Gladbeck, Grevenbroich, Gütersloh, Herford, Herten, Iserlohn, Kerpen, Lippstadt, Lüdenscheid, Lünen, Marl, Minden, Moers, Neuss, Paderborn, Ratingen, Recklinghausen, Rheine, Siegen, Troisdorf, Unna, Velbert, Viersen, Wesel, Witten"

kommunen = {k.strip().lower() for k in grosse_kommunen.split(",")}
pattern = "|".join(map(re.escape, kommunen))
name_l = df["Name"].astype(str).str.lower()
mask_special = name_l.str.contains("jugendamt", na=False) & name_l.str.contains(pattern, na=False)
df.loc[mask_special, "kreis_code_tmp"] = df.loc[mask_special, "Unnamed: 1"].astype(str).str[2:7]
large_codes = set(df.loc[mask_special, "kreis_code_tmp"].dropna())

df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")

df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]
df = df[~df["Name"].str.contains("Kreisjugendamt", case=False, na=False)]


# 3) Flag nur bei exakt passender Kreis-Zeile setzen
df["Kreis mit großer Gemeinde"] = (
    df["Unnamed: 1"].astype(str).isin(large_codes)
)

# Object-Types parsen
df['Anzahl 35a Hilfen'] = pd.to_numeric(df['Anzahl 35a Hilfen'], errors='coerce').astype("Int64")
df['Insgesamt'] = pd.to_numeric(df['Insgesamt'], errors='coerce').astype("Int64")

In [5]:
# fehlende Werte finden
print(df[df["Anzahl 35a Hilfen"].isna()])
print(df[df["Insgesamt"].isna()])

   Unnamed: 0 Unnamed: 1                     Name  Insgesamt Insgesamt.1  \
6        2024      05113       Essen, krfr. Stadt       3894        2671   
54       2024      05313      Aachen, krfr. Stadt       <NA>           -   
66       2024      05354            Aachen, Kreis       <NA>           -   

   Hilfe zur Erziehung § 27 SGB VIII Hilfe zur Erziehung § 27 SGB VIII.1  \
6                                  -                                   -   
54                                 -                                   -   
66                                 -                                   -   

   Erziehungsberatung § 28 SGB VIII Erziehungsberatung § 28 SGB VIII.1  \
6                              3894                               2671   
54                                -                                  -   
66                                -                                  -   

   Soziale Gruppenarbeit § 29 SGB VIII  ... Vollzeitpflege § 33 SGB VIII  \
6                

In [6]:
# Aachen ist mittlerweile Städteregion, daher wird alles außer Städteregion gelöscht
df["Name"] = df["Name"].str.strip()
df = df[df["Name"] != "Aachen, krfr. Stadt"]
df = df[df["Name"] != "Aachen, Kreis"]

In [7]:
# Typ (Kreis/kreisfreie Stadt/großer Kreis) extrahieren
kreisfreie_keywords = ["krfr. Stadt", "kreisfreie Stadt", "Stadt"]
kreis_keywords = ["Kreis", "kreis"]

def extract(raw: str, grosser_kreis: bool):
    raw = str(raw).strip()
    grosser_kreis = bool(grosser_kreis)

    if any(k in raw for k in kreisfreie_keywords):
        return raw, "Kreisfreie Stadt"

    if any(k in raw for k in kreis_keywords):
        return raw, "Großer Kreis" if grosser_kreis else "Kleiner Kreis"

    return raw, "Unbekannt"

df[["Name", "Typ 1"]] = df.apply(
    lambda r: pd.Series(extract(r["Name"], r["Kreis mit großer Gemeinde"])),
    axis=1
)

df["Typ 2"] = df["Typ 1"].replace(
    {"Großer Kreis": "Kreis", "Kleiner Kreis": "Kreis"})

print((df["Typ 1"] == "Unbekannt").sum())
print(df)

0
    Unnamed: 0 Unnamed: 1                                         Name  \
4         2024      05111                      Düsseldorf, krfr. Stadt   
5         2024      05112                        Duisburg, krfr. Stadt   
6         2024      05113                           Essen, krfr. Stadt   
7         2024      05114                         Krefeld, krfr. Stadt   
8         2024      05116                 Mönchengladbach, krfr. Stadt   
9         2024      05117             Mülheim an der Ruhr, krfr. Stadt   
10        2024      05119                      Oberhausen, krfr. Stadt   
11        2024      05120                       Remscheid, krfr. Stadt   
12        2024      05122                        Solingen, krfr. Stadt   
13        2024      05124                       Wuppertal, krfr. Stadt   
14        2024      05154                                 Kleve, Kreis   
21        2024      05158                              Mettmann, Kreis   
32        2024      05162           

In [8]:
# Strings parsen und anpassen
df = clean_and_sort(df, "Typ 1", "Typ 2", "Insgesamt", "Anzahl 35a Hilfen")

In [10]:
df.loc[df["Name"] == "Aachen", "Typ 1"] ="Städteregion"
df.loc[df["Name"] == "Aachen", "Typ 2"] = "Städteregion"
df.loc[df["Name"] == "Essen", "Anzahl 35a Hilfen"] = 0 #dummy

In [11]:
validate_df(
    df,
    not_null=["Insgesamt", "Anzahl 35a Hilfen"],
    non_negative=["Insgesamt", "Anzahl 35a Hilfen"],
    numeric=["Insgesamt", "Anzahl 35a Hilfen"],
    key_cols=["Name"],
)

DataFrame: alle Checks bestanden


[]

In [12]:
# saubere Tabelle abspeichern
df.to_csv("../data/processed/hilfen_2024.csv", index=False)